## Homework: Image Segmentation – Parameter Analysis & Model Replication

Citations:

Code Formatted in Claude Code
Claude Code Chat: https://claude.ai/share/47431a7a-4e01-4fbd-8311-51d86231a139
VGG Layers Taken from Online Sources: https://www.kaggle.com/code/blurredmachine/vggnet-16-architecture-a-complete-guide

## VGG-16 Backbone — Parameter Count

| Block | Parameters |
|-------|------------|
| Block 1 | 38,720 |
| Block 2 | 221,440 |
| Block 3 | 1,475,328 |
| Block 4 | 5,899,776 |
| Block 5 | 7,079,424 |
| **VGG-16 Total** | **14,714,688** |

## Convolutional FC Layers — Parameter Count

| Layer | Originally | $C_{in}$ | $C_{out}$ | $K$ | Formula | Parameters |
|-------|-----------|----------|-----------|-----|---------|------------|
| conv6 | fc6 (25088→4096) | 512 | 4096 | 7 | $512 \times 4096 \times 7 \times 7 + 4096$ | 102,764,544 |
| conv7 | fc7 (4096→4096) | 4096 | 4096 | 1 | $4096 \times 4096 \times 1 \times 1 + 4096$ | 16,781,312 |
| **Conv. FC Total** | | | | | | **119,545,856** |

## Score Layer — Parameter Count

| Layer | $C_{in}$ | $C_{out}$ | $K$ | Formula | Parameters |
|-------|----------|-----------|-----|---------|------------|
| score_conv | 4096 | 21 | 1 | $4096 \times 21 \times 1 \times 1 + 21$ | 86,037 |

## FCN-32s Total Parameters

| | Parameters |
|---|------------|
| VGG-16 Backbone | 14,714,688 |
| Conv. FC Layers | 119,545,856 |
| Score Layer | 86,037 |
| **Total** | **134,346,581** |

The paper rounds to "134M." Our count of 134,346,581 is consistent. The ~346K gap is attributable to rounding and minor implementation details 

In [3]:
import torch
import torchvision
import torchvision.transforms as T
import torchvision.datasets as datasets
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


1. Load the VOC 2012 validation set with image normalization and tensor conversion.

In [4]:
import torchvision.datasets as datasets

# Load PASCAL VOC 2012 validation set
dataset = datasets.VOCSegmentation(
    root='/home/suwaraich/CS_343_AI/HW3',
    year='2012', 
    image_set='val',
    download=True
)

print(f"Dataset size: {len(dataset)}")
print(f"First sample: {dataset[0]}")  # Returns (image, segmentation_mask)

Dataset size: 1449
First sample: (<PIL.Image.Image image mode=RGB size=500x366 at 0x714F15257EF0>, <PIL.PngImagePlugin.PngImageFile image mode=P size=500x366 at 0x714F15282630>)


2. Load the pretrained FCN-ResNet101 model from torchvision.models for semantic segmentation.

In [5]:
model = torchvision.models.segmentation.fcn_resnet101(pretrained=True)
model.eval()
print("Model loaded")


/home/suwaraich/CS_343_AI/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/suwaraich/CS_343_AI/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FCN_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=FCN_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model loaded


3. Run the pretrained model on all validation images and store predictions and ground truth masks.

In [7]:
import torchvision.transforms as T

transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

all_preds = []
all_gts   = []

start = time.time()

for i in range(len(dataset)):
    image, mask = dataset[i]
    
    image = transform(image)
    mask = np.array(mask)

    with torch.no_grad():
        output = model(image.unsqueeze(0))['out']

    all_preds.append(output.argmax(dim=1))
    all_gts.append(torch.as_tensor(mask, dtype=torch.long).unsqueeze(0))

    if (i + 1) % 100 == 0:
        print(f"{i+1}/{len(dataset)}")

print(f"Inference done in {time.time() - start:.1f}s")


100/1449
200/1449
300/1449
400/1449
500/1449
600/1449
700/1449
800/1449
900/1449
1000/1449
1100/1449
1200/1449
1300/1449
1400/1449
Inference done in 382.1s


4. Calculate Mean Intersection-over-Union (mIoU) and pixel accuracy from stored predictions.

In [10]:
NUM_CLASSES = 21
intersection = np.zeros(NUM_CLASSES)
union = np.zeros(NUM_CLASSES)
correct = 0
total = 0

for pred, gt in zip(all_preds, all_gts):
    pred = pred.cpu().numpy()
    gt = gt.cpu().numpy()
    
    valid = gt != 255
    correct += (pred[valid] == gt[valid]).sum()
    total += valid.sum()
    
    for c in range(NUM_CLASSES):
        inter = ((pred == c) & (gt == c) & valid).sum()
        uni = ((pred == c) | (gt == c)) & valid
        intersection[c] += inter
        union[c] += uni.sum()

iou = intersection / (union + 1e-6)
miou = iou.mean()
pixel_acc = correct / total

print(f"mIoU:           {miou:.4f} ({miou*100:.2f}%)")
print(f"Pixel Accuracy: {pixel_acc:.4f} ({pixel_acc*100:.2f}%)")

mIoU:           0.7511 (75.11%)
Pixel Accuracy: 0.9442 (94.42%)


5. Measure inference time per image and peak GPU memory consumption.

In [12]:
total_time = time.time() - start  # from Cell 4 start
total_images = len(dataset)

print(f"Total inference time: {total_time:.1f}s")
print(f"Avg per image:        {total_time/total_images*1000:.1f}ms")


Total inference time: 726.0s
Avg per image:        501.1ms


6. Display per-class IoU scores and visualize confusion matrix for the first 3 classes (Background, Aeroplane, Bicycle).

In [15]:
print("\nPer-Class IoU:")
for c in range(21):
    if union[c] > 0:
        print(f"  Class {c:2d}: {iou[c]*100:.2f}%")

print("\nConfusion Matrix (Classes 0, 1, 2):")
from sklearn.metrics import confusion_matrix

all_preds_flat = np.concatenate([p.cpu().numpy().flatten() for p in all_preds])
all_gts_flat = np.concatenate([g.cpu().numpy().flatten() for g in all_gts])

valid_flat = all_gts_flat != 255
cm = confusion_matrix(all_gts_flat[valid_flat], all_preds_flat[valid_flat], labels=[0, 1, 2])
print(cm)
print("Rows: Ground Truth | Columns: Prediction")


Per-Class IoU:
  Class  0: 93.79%
  Class  1: 87.60%
  Class  2: 39.80%
  Class  3: 86.28%
  Class  4: 68.42%
  Class  5: 56.48%
  Class  6: 93.64%
  Class  7: 76.20%
  Class  8: 87.66%
  Class  9: 47.51%
  Class 10: 83.41%
  Class 11: 63.34%
  Class 12: 80.56%
  Class 13: 84.52%
  Class 14: 79.77%
  Class 15: 89.51%
  Class 16: 55.93%
  Class 17: 83.92%
  Class 18: 59.40%
  Class 19: 84.36%
  Class 20: 75.28%

Confusion Matrix (Classes 0, 1, 2):
[[174587449     45012    845316]
 [   156640   1764647         0]
 [    93890         0    689205]]
Rows: Ground Truth | Columns: Prediction


### Task 2B: Replicate the Architecture from Scratch

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

1. Define the complete FCN-8s model using torch.nn.Module with VGG-16 backbone, skip connections, and upsampling layers.

In [20]:
class FCN8s(nn.Module):
    def __init__(self, num_classes=21):
        super(FCN8s, self).__init__()
        
        # VGG16 backbone
        self.conv1_1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, stride=2)
        
        self.conv2_1 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, stride=2)
        
        self.conv3_1 = nn.Conv2d(128, 256, 3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, 3, padding=1)
        self.pool3 = nn.MaxPool2d(2, stride=2)
        
        self.conv4_1 = nn.Conv2d(256, 512, 3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, 3, padding=1)
        self.pool4 = nn.MaxPool2d(2, stride=2)
        
        self.conv5_1 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, 3, padding=1)
        self.pool5 = nn.MaxPool2d(2, stride=2)
        
        # FC as conv (with padding to preserve spatial dims)
        self.fc6 = nn.Conv2d(512, 4096, 7, padding=3)
        self.fc7 = nn.Conv2d(4096, 4096, 1)
        
        # Score layer
        self.score_pool5 = nn.Conv2d(4096, num_classes, 1)
        self.score_pool4 = nn.Conv2d(512, num_classes, 1)
        self.score_pool3 = nn.Conv2d(256, num_classes, 1)
        
        # Upsample
        self.upsample_2x = nn.ConvTranspose2d(num_classes, num_classes, 4, stride=2, padding=1)
        self.upsample_8x = nn.ConvTranspose2d(num_classes, num_classes, 16, stride=8, padding=4)

    def forward(self, x):
        # Encoder
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.pool1(x)
        
        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = self.pool2(x)
        
        x = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(x))
        x = F.relu(self.conv3_3(x))
        pool3 = self.pool3(x)
        
        x = F.relu(self.conv4_1(pool3))
        x = F.relu(self.conv4_2(x))
        x = F.relu(self.conv4_3(x))
        pool4 = self.pool4(x)
        
        x = F.relu(self.conv5_1(pool4))
        x = F.relu(self.conv5_2(x))
        x = F.relu(self.conv5_3(x))
        pool5 = self.pool5(x)
        
        # FC layers
        x = F.relu(self.fc6(pool5))
        x = F.relu(self.fc7(x))
        
        # Decode
        score5 = self.score_pool5(x)
        score4 = self.score_pool4(pool4)
        score3 = self.score_pool3(pool3)
        
        # Skip connections
        upscore2 = self.upsample_2x(score5)
        fuse4 = upscore2 + score4
        
        upscore_pool4 = self.upsample_2x(fuse4)
        fuse3 = upscore_pool4 + score3
        
        # Final upsample
        out = self.upsample_8x(fuse3)
        
        return out

model = FCN8s(num_classes=21)
print("Model created")

Model created


2. Count total trainable parameters and compare with official paper specifications to verify correct implementation.

In [21]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Count by layer type
conv_params = sum(p.numel() for name, p in model.named_parameters() if 'conv' in name)
fc_params = sum(p.numel() for name, p in model.named_parameters() if 'fc' in name)
print(f"Conv params:      {conv_params:,}")
print(f"FC params:        {fc_params:,}")

Total parameters: 134,482,745
Conv params:      14,714,688
FC params:        119,545,856


3. Validate that the model produces correct output shape (1, 21, 224, 224) for a sample input.

In [22]:
dummy_input = torch.randn(1, 3, 224, 224)
output = model(dummy_input)
print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Expected:     (1, 21, 224, 224)")

Input shape:  torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 21, 224, 224])
Expected:     (1, 21, 224, 224)


4. Apply Kaiming normal initialization to convolutional layers as specified in the FCN paper.

In [23]:
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.ConvTranspose2d):
        nn.init.normal_(m.weight, 0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

model.apply(init_weights)
print("Weights initialized")

Weights initialized


5. Configure SGD optimizer and CrossEntropyLoss for training, and move model to GPU/CPU device.

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss(ignore_index=255)

print(f"Device: {device}")
print("Ready to train")

Device: cuda
Ready to train


6. Configure SGD optimizer and CrossEntropyLoss for training, and move model to GPU/CPU device.

In [34]:
from torch.utils.data import DataLoader

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def target_transform(mask):
    mask = T.functional.resize(mask, (224, 224), interpolation=T.InterpolationMode.NEAREST)
    return torch.as_tensor(np.array(mask), dtype=torch.long)

dataset = datasets.VOCSegmentation(
    root='/home/suwaraich/CS_343_AI/HW3',
    year='2012',
    image_set='train',
    download=False,
    transform=transform,
    target_transform=target_transform
)

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
print(f"Dataset size: {len(dataset)}")

Dataset size: 1464


7. Train the custom FCN-8s model on 50 training batches, computing and logging loss at regular intervals.

In [35]:
model.train()
total_loss = 0

for i, (image, mask) in enumerate(loader):
    if i >= 50:  # Just 50 batches for demo
        break
    
    image = image.to(device)
    mask = mask.to(device).long()
    
    output = model(image)
    loss = criterion(output, mask)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    total_loss += loss.item()
    
    if (i + 1) % 10 == 0:
        print(f"Batch {i+1}: loss = {loss.item():.4f}")

print(f"Average loss: {total_loss / 50:.4f}")

Batch 10: loss = 2.6056
Batch 20: loss = 1.6024
Batch 30: loss = 2.8483
Batch 40: loss = 1.5144
Batch 50: loss = 1.0426
Average loss: 1.9523


8. Compute pixel accuracy on 20 test batches to measure model performance.

In [36]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for i, (image, mask) in enumerate(loader):
        if i >= 20:  # Test on 20 batches
            break
        
        image = image.to(device)
        mask = mask.to(device).long()
        
        output = model(image)
        pred = output.argmax(dim=1)
        
        valid = mask != 255
        correct += (pred[valid] == mask[valid]).sum().item()
        total += valid.sum().item()

accuracy = correct / total
print(f"Test Pixel Accuracy: {accuracy*100:.2f}%")

Test Pixel Accuracy: 73.81%
